# Donor Retention Predictive

## Problem Framing

**Business question:** Who is at risk of donor lapse?

**Operational decision supported:** Prioritize weekly donor-retention outreach and stewardship follow-up.

**Primary users:** fundraisers, donor engagement leads

**Target summary:** Current predictive label: `label_lapsed_180d` from supporter snapshots.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='donor_retention',
    dataset_name='supporter_features',
    predictive_impl='donor_retention',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,supporter_id,supporter_type,display_name,organization_name,first_name,last_name,relationship_type,region,country,email,...,total_allocated_amount,allocation_safehouse_diversity,allocation_program_diversity,in_kind_item_line_count,in_kind_quantity_total,in_kind_category_diversity,in_kind_estimated_value_total,supporter_tenure_days,has_any_donation,label_lapsed_180d
0,1,SocialMediaAdvocate,Mila Alvarez,NaN,Mila,Alvarez,Local,Luzon,Philippines,mila-alvarez@smart.com.ph,...,8540.11,9.0,5.0,4.0,62.0,3.0,1162.86,1520,True,0
1,2,Volunteer,Aria Brown,NaN,Aria,Brown,Local,Mindanao,Philippines,aria-brown@pldt.net.ph,...,3877.37,6.0,3.0,1.0,12.0,1.0,388.92,1515,True,1
2,3,MonetaryDonor,Noah Chen,NaN,Noah,Chen,Local,Luzon,Philippines,noah-chen@globe.com.ph,...,12445.41,8.0,6.0,8.0,100.0,5.0,5514.93,1510,True,0
3,4,MonetaryDonor,Liam Diaz,NaN,Liam,Diaz,PartnerOrganization,Mindanao,Philippines,liam-diaz@globe.com.ph,...,9847.12,7.0,6.0,4.0,50.0,3.0,813.10,1505,True,0
4,5,InKindDonor,Emma Evans,NaN,Emma,Evans,PartnerOrganization,Mindanao,Philippines,emma-evans@yahoo.com.ph,...,4751.16,4.0,3.0,0.0,0.0,0.0,0.00,1500,True,0
5,6,MonetaryDonor,Olivia Ford,NaN,Olivia,Ford,International,Visayas,USA,olivia-ford@yahoo.com,...,8457.86,9.0,4.0,1.0,8.0,1.0,618.13,1495,True,0
6,7,MonetaryDonor,Ethan Garcia,NaN,Ethan,Garcia,International,Mindanao,USA,ethan-garcia@aol.com,...,5609.54,7.0,6.0,5.0,51.0,4.0,3007.24,1490,True,0
7,8,InKindDonor,Isla Hernandez,NaN,Isla,Hernandez,Local,Mindanao,Philippines,isla-hernandez@yahoo.com.ph,...,8796.67,8.0,6.0,2.0,38.0,2.0,1977.65,1485,True,0
8,9,Volunteer,Sophia Ibrahim,NaN,Sophia,Ibrahim,Local,Luzon,Philippines,sophia-ibrahim@globe.com.ph,...,9686.99,6.0,4.0,2.0,38.0,2.0,1669.44,1480,True,0
9,10,Volunteer,Lucas Jones,NaN,Lucas,Jones,International,Luzon,Singapore,lucas-jones@gmail.com,...,3353.54,4.0,4.0,4.0,89.0,3.0,1705.21,1475,True,0


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_lapsed_180d` from supporter snapshots.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('donor_retention')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'logistic_regression',
 'train_rows': 47,
 'test_rows': 12,
 'target_col': 'label_lapsed_180d',
 'split_col': 'created_at',
 'selection_metric': 'average_precision',
 'cutoff_date': None,
 'task_type': 'classification',
 'sample_count': 12.0,
 'positive_count': 6.0,
 'positive_rate': 0.5,
 'accuracy': 0.75,
 'balanced_accuracy': 0.75,
 'precision': 0.8,
 'recall': 0.6666666666666666,
 'f1': 0.7272727272727273,
 'roc_auc': 0.8333333333333333,
 'average_precision': 0.8634920634920633}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,donor_retention,logistic_regression,12.0,6.0,0.5,0.750000,0.750000,0.8,0.666667,0.727273,0.833333,0.863492,NaN,NaN,NaN
1,donor_retention,random_forest_classifier,12.0,6.0,0.5,0.583333,0.583333,0.6,0.500000,0.545455,0.555556,0.660354,NaN,NaN,NaN
2,donor_retention,dummy_classifier,12.0,6.0,0.5,0.500000,0.500000,0.0,0.000000,0.000000,0.500000,0.500000,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,donor_retention,random_forest_classifier,3.0,1.581139,9.4,0.547723,3.0,0.0,0.32,0.018257,...,0.162078,0.833333,0.154897,5,NaN,NaN,NaN,NaN,NaN,NaN
1,donor_retention,logistic_regression,3.0,1.581139,9.4,0.547723,3.0,0.0,0.32,0.018257,...,0.069778,0.701905,0.117629,5,NaN,NaN,NaN,NaN,NaN,NaN
2,donor_retention,dummy_classifier,3.0,1.581139,9.4,0.547723,3.0,0.0,0.32,0.018257,...,0.000000,0.320000,0.018257,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to fundraisers, donor engagement leads?


## Deployment Notes

Recommended shared widgets:

* `ranked_table_widget`
* `risk_badge_widget`
* `recommendation_panel`

Deployment checklist:

* Surface the score on donor profiles and fundraiser queue views.
* Pair the ranked list with a recommendation panel for retention action planning.

Standard endpoint pattern:

* `POST /ml/predict/donor_retention`
* `POST /ml/score-batch/donor_retention`
